In [1]:
%pip install ucimlrepo --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

ds = fetch_ucirepo(id=601)  # 601 = AI4I 2020 Predictive Maintenance
df_raw = pd.concat([ds.data.features, ds.data.targets], axis=1)
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

Shape: (10000, 12)
Columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
# Some versions include units, e.g. "Air temperature [K]". Remove them.
df = df_raw.copy()
df.columns = [c.split(" [")[0].strip() for c in df.columns]

rename_map = {
    "Type": "type",
    "Air temperature": "air_temp_k",
    "Process temperature": "process_temp_k",
    "Rotational speed": "rpm",
    "Torque": "torque_nm",
    "Tool wear": "tool_wear_min",
    "Machine failure": "machine_failure",
}
df = df.rename(columns=rename_map)[list(rename_map.values())]

# Drops the failure-mode columns TWF, HDF, PWF, OSF, RNF.
# Those failure modes ARE the failure, so keeping them would leak the answer.
df.head()

,type,air_temp_k,process_temp_k,rpm,torque_nm,tool_wear_min,machine_failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [4]:
print("Missing values per column:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)
print("\nTarget balance (share of rows):")
print(df["machine_failure"].value_counts(normalize=True).round(4))

Missing values per column:
type               0
air_temp_k         0
process_temp_k     0
rpm                0
torque_nm          0
tool_wear_min      0
machine_failure    0
dtype: int64

Data types:
type                object
air_temp_k         float64
process_temp_k     float64
rpm                  int64
torque_nm          float64
tool_wear_min        int64
machine_failure      int64
dtype: object

Target balance (share of rows):
0    0.9661
1    0.0339
Name: machine_failure, dtype: float64


In [5]:
df.to_csv("../data/ai4i_clean.csv", index=False)
print("Saved", df.shape, "to data/ai4i_clean.csv")

Saved (10000, 7) to data/ai4i_clean.csv


In [6]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "f3366df5-0550-438c-b31e-e437667ba4b0"
RESOURCE_GROUP = "rg-mlab1-iram15"
WORKSPACE = "mlw-lab1-iram15"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

Connected to: mlw-lab1-iram15


In [7]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

csv_asset = Data(
    name="ai4i-maintenance-csv",
    version="1",
    type=AssetTypes.URI_FILE,
    path="../data/ai4i_clean.csv",
    description="AI4I 2020 predictive maintenance, cleaned: 6 features + machine_failure",
)
csv_asset = ml_client.data.create_or_update(csv_asset)
print("Registered:", csv_asset.name, "version", csv_asset.version)

Uploading ai4i_clean.csv (< 1 MB): 100%|██████████| 295k/295k [00:00<00:00, 10.5MB/s]




Registered: ai4i-maintenance-csv version 1


In [8]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

csv_asset = Data(
    name="ai4i-maintenance-csv",
    version="1",
    type=AssetTypes.URI_FILE,
    path="../data/ai4i_clean.csv",
    description="AI4I 2020 predictive maintenance, cleaned: 6 features + machine_failure",
)
csv_asset = ml_client.data.create_or_update(csv_asset)
print("Registered:", csv_asset.name, "version", csv_asset.version)

HttpResponseError: (UserError) A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Code: UserError
Message: A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Additional Information:Type: ComponentName
Info: {
    "value": "managementfrontend"
}Type: Correlation
Info: {
    "value": {
        "operation": "22b71af61f9db8275e3868c69ff20bfb",
        "request": "bfe8ba3c75dc5ad3"
    }
}Type: Environment
Info: {
    "value": "eastus"
}Type: Location
Info: {
    "value": "eastus"
}Type: Time
Info: {
    "va

In [9]:
import os, shutil

os.makedirs("../data/mltable_ai4i", exist_ok=True)
shutil.copy("../data/ai4i_clean.csv", "../data/mltable_ai4i/ai4i_clean.csv")

mltable_yaml = """paths:
  - file: ./ai4i_clean.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: utf8
      header: all_files_same_headers
  - convert_column_types:
      - columns: [air_temp_k, process_temp_k, torque_nm]
        column_type: float
      - columns: [rpm, tool_wear_min, machine_failure]
        column_type: integer
      - columns: [type]
        column_type: string
"""
with open("../data/mltable_ai4i/MLTable", "w") as f:
    f.write(mltable_yaml)

print(os.listdir("../data/mltable_ai4i"))

['ai4i_clean.csv', 'MLTable']


In [10]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

table_asset = Data(
    name="ai4i-maintenance-table",
    version="1",
    type=AssetTypes.MLTABLE,
    path="../data/mltable_ai4i",
    description="AI4I 2020 predictive maintenance as MLTable for AutoML",
)
table_asset = ml_client.data.create_or_update(table_asset)
print("Registered:", table_asset.name, "version", table_asset.version)

MlException: 
[37m
[30m
1) One or more fields are invalid[39m[39m

Details: 

[31m(x) 'integer' is not one of ['boolean', 'float', 'string', 'int', 'stream_info']
Invalid MLTable metadata schema:
{
  "type": "string",
  "enum": [
    "boolean",
    "float",
    "string",
    "int",
    "stream_info"
  ]
}[39m

Resolutions: 
1) Double-check that all specified parameters are of the correct types and formats prescribed by the Data schema.
If using the CLI, you can also check the full log in debug mode for more details by adding --debug to the end of your command

Additional Resources: The easiest way to author a yaml specification file is using IntelliSense and auto-completion Azure ML VS code extension provides: [36mhttps://code.visualstudio.com/docs/datascience/azure-machine-learning.[39m To set up VS Code, visit [36mhttps://learn.microsoft.com/azure/machine-learning/how-to-setup-vs-code[39m


In [11]:
mltable_yaml = """paths:
  - file: ./ai4i_clean.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: utf8
      header: all_files_same_headers
  - convert_column_types:
      - columns: [air_temp_k, process_temp_k, torque_nm]
        column_type: float
      - columns: [rpm, tool_wear_min, machine_failure]
        column_type: int
      - columns: [type]
        column_type: string
"""
with open("../data/mltable_ai4i/MLTable", "w") as f:
    f.write(mltable_yaml)

print(open("../data/mltable_ai4i/MLTable").read())

paths:
  - file: ./ai4i_clean.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: utf8
      header: all_files_same_headers
  - convert_column_types:
      - columns: [air_temp_k, process_temp_k, torque_nm]
        column_type: float
      - columns: [rpm, tool_wear_min, machine_failure]
        column_type: int
      - columns: [type]
        column_type: string



In [12]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

table_asset = Data(
    name="ai4i-maintenance-table",
    version="1",
    type=AssetTypes.MLTABLE,
    path="../data/mltable_ai4i",
    description="AI4I 2020 predictive maintenance as MLTable for AutoML",
)
table_asset = ml_client.data.create_or_update(table_asset)
print("Registered:", table_asset.name, "version", table_asset.version)

Uploading mltable_ai4i (0.3 MBs): 100%|██████████| 295336/295336 [00:00<00:00, 7800363.76it/s]




Registered: ai4i-maintenance-table version 1


In [13]:
cd ~/cloudfiles/code/Users/*/azureml-knn-maintenance

[Errno 2] No such file or directory: '/home/azureuser/cloudfiles/code/Users/*/azureml-knn-maintenance'
/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-iram15-lab1/code/Users/iramshahzadishahzadi892/azureml-knn-maintenance/notebooks
